In [234]:
#import des librairies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [235]:
df_effectifs = pd.read_csv(
    "../data/raw/Sante/effectifs.csv", sep=';', 
)
df_effectifs.head()

,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
0,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,1,971,280.0,17040,1.643,"2,3",de 50 à 54 ans,femmes,48.0
1,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,3,999,50.0,6090,0.771,"2,3",de 50 à 54 ans,femmes,48.0
2,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,4,974,390.0,32630,1.205,"2,3",de 50 à 54 ans,femmes,48.0
3,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,11,77,350.0,48230,0.732,"2,3",de 50 à 54 ans,femmes,48.0
4,2020,Maladies psychiatriques,Troubles psychotiques,Troubles psychotiques,PSY_PSC_IND,50-54,2,11,91,400.0,43370,0.927,"2,3",de 50 à 54 ans,femmes,48.0


In [236]:
df_effectifs.info()

<class 'pandas.DataFrame'>
RangeIndex: 5796000 entries, 0 to 5795999
Data columns (total 16 columns):
 #   Column              Dtype  
---  ------              -----  
 0   annee               int64  
 1   patho_niv1          str    
 2   patho_niv2          str    
 3   patho_niv3          str    
 4   top                 str    
 5   cla_age_5           str    
 6   sexe                int64  
 7   region              int64  
 8   dept                str    
 9   Ntop                float64
 10  Npop                int64  
 11  prev                float64
 12  Niveau prioritaire  str    
 13  libelle_classe_age  str    
 14  libelle_sexe        str    
 15  tri                 float64
dtypes: float64(3), int64(4), str(9)
memory usage: 707.5 MB


In [237]:
#Quelles sont les valeurs uniques pour les pathologies de niveau 1
print(df_effectifs["patho_niv1"].unique())

<StringArray>
[                                                                                         'Maladies psychiatriques',
                                                           'Maladies respiratoires chroniques (hors mucoviscidose)',
                                                                'Maladies inflammatoires ou rares ou infection VIH',
                                                                                           'Maladies neurologiques',
 'Pas de pathologie repérée, traitement, maternité, hospitalisation ou traitement antalgique ou anti-inflammatoire',
                                                                                   'Total consommants tous régimes',
                                                                                                          'Cancers',
                                                                                                          'Diabète',
                                                  

1. Filtrage sur patho_niv1
Le fichier est assez lourd, pour ne pas avoir à manipuler tout les données qui ne seront pas utiles pour notre projet, je commence par faire 2 df differentes:
 un avec uniquement les maladies respiratoires chroniques (b), et l'autre qui contient les hospitalisations pour Covid-19, Traitement antalgique ou anti-inflammatoire et Cancers (a)

In [238]:
# 1. Définir les catégories à explorer
pathologies = [
    "Maladies respiratoires chroniques (hors mucoviscidose)",
    "Hospitalisation pour Covid-19",
    "Traitements antalgiques ou anti-inflammatoires (hors pathologies, traitements, maternité ou hospitalisations)",
    "Cancers"
]
pathologie_respi = [
    "Maladies respiratoires chroniques (hors mucoviscidose)"
]

# 1. a) creation nouvelle df filtré pour 4 categories selectionnées
df_effectifs_filtre_patho = df_effectifs[
    df_effectifs["patho_niv1"].isin(pathologies)
].copy()



# 1.b) creation nouvelle df filtré pour les maladies respiratoires uniquement

df_respi = df_effectifs[
    df_effectifs["patho_niv1"].isin(pathologie_respi)
]


In [239]:
#Exportation des 2 fichiers
df_effectifs_filtre_patho.to_csv("df_effectifs_filtre_patho.csv", sep=",", index=False)

df_respi.to_csv("df_respi", sep=",", index=False)

2. Nettoyage des colonnes redondantes
Je continue le preprocessing avec df_respi.

a) Niveau pathologie et categorie top: la catégorie "Maladie respiratoires chroniques" regroupe plusieurs maladies respiratoires chroniques : elle ne permet pas d’isoler précisément l’asthme, l’allergie ou la bronchiolite.

In [240]:
print("Original :", df_effectifs.shape)

print("Respi:", df_respi.shape)

df_respi.head(5)

Original : (5796000, 16)
Respi: (75600, 16)


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
1259,2020,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,RES_CAT_EXC,00-04,1,2,972,470.0,8310,5.679,"1,2,3",de 0 à 4 ans,hommes,10.0
1260,2020,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,RES_CAT_EXC,00-04,1,6,999,90.0,8130,1.046,"1,2,3",de 0 à 4 ans,hommes,10.0
1261,2020,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,RES_CAT_EXC,00-04,1,11,92,3080.0,52210,5.899,"1,2,3",de 0 à 4 ans,hommes,10.0
1262,2020,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,RES_CAT_EXC,00-04,1,11,95,2370.0,45900,5.168,"1,2,3",de 0 à 4 ans,hommes,10.0
1263,2020,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,RES_CAT_EXC,00-04,1,11,999,21150.0,393510,5.373,"1,2,3",de 0 à 4 ans,hommes,10.0


In [241]:
# Verification des souscategories des pathologies

colonnes = ["patho_niv1", "patho_niv2", "patho_niv3", "top"]

for col in colonnes:
    print(f"\n--- {col} ---")
    print(df_respi[col].value_counts(dropna=False))


--- patho_niv1 ---
patho_niv1
Maladies respiratoires chroniques (hors mucoviscidose)    75600
Name: count, dtype: int64

--- patho_niv2 ---
patho_niv2
Maladies respiratoires chroniques (hors mucoviscidose)    75600
Name: count, dtype: int64

--- patho_niv3 ---
patho_niv3
Maladies respiratoires chroniques (hors mucoviscidose)    75600
Name: count, dtype: int64

--- top ---
top
RES_CAT_EXC    75600
Name: count, dtype: int64


In [242]:
# Verification si il y a plusieurs sous-categories lié au patho_niv1
df_respi[colonnes].drop_duplicates()

,patho_niv1,patho_niv2,patho_niv3,top
1259,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,Maladies respiratoires chroniques (hors mucovi...,RES_CAT_EXC


--> Entre les 4 colonnes patho_niv1, patho_niv2, patho_niv3 et top les informations sont redontandtes, je vais garder la categorie "top" qui contient le code et creer un colonne "pathologie" avec un nom plus court pour la pathologie  et supprimer le reste (patho_niv1, patho_niv2, patho_niv3)



1. Nettoyage
a) Supression colonnes patho_niv1, patho_niv2, patho_niv3, creation colonne pathologie
b) --> libellées classe d_age et libelle_sexe car ils sont redondant et reconstituables.

Je garde la colonne "region" à cet etape pour pas perdre l'info des aggregations territoriaux

In [243]:
#creation d'un copy de travail
df_respi_travail = df_respi.copy()

#creation d'une colonnne "pathologie"
df_respi_travail["pathologie"] = "Maladie_Respi_Chronique"

#Supresson des colonnes: "libelle_classe_age","libelle_sexe","patho_niv1","patho_niv2", "patho_niv3", "Niveau prioritaire", "tri"
df_respi_travail = df_respi_travail.drop(columns=["libelle_classe_age", "libelle_sexe", "patho_niv1", "patho_niv2", "patho_niv3","Niveau prioritaire", "tri" ])

display(df_respi_travail.info())
display(df_respi_travail.head(5))


<class 'pandas.DataFrame'>
Index: 75600 entries, 1259 to 5759926
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   annee       75600 non-null  int64  
 1   top         75600 non-null  str    
 2   cla_age_5   75600 non-null  str    
 3   sexe        75600 non-null  int64  
 4   region      75600 non-null  int64  
 5   dept        75600 non-null  str    
 6   Ntop        75421 non-null  float64
 7   Npop        75600 non-null  int64  
 8   prev        75421 non-null  float64
 9   pathologie  75600 non-null  str    
dtypes: float64(2), int64(4), str(4)
memory usage: 6.3 MB


None

,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie
1259,2020,RES_CAT_EXC,00-04,1,2,972,470.0,8310,5.679,Maladie_Respi_Chronique
1260,2020,RES_CAT_EXC,00-04,1,6,999,90.0,8130,1.046,Maladie_Respi_Chronique
1261,2020,RES_CAT_EXC,00-04,1,11,92,3080.0,52210,5.899,Maladie_Respi_Chronique
1262,2020,RES_CAT_EXC,00-04,1,11,95,2370.0,45900,5.168,Maladie_Respi_Chronique
1263,2020,RES_CAT_EXC,00-04,1,11,999,21150.0,393510,5.373,Maladie_Respi_Chronique


1. Nettoyage
c) Supprimer les annees pas retenus par l'equipe, garder la periode 2021 -2024


In [244]:
# Garder uniquement les années choisies
df_annees = df_respi_travail["annee"].isin([2021, 2022, 2023, 2024])

df_respi_travail = df_respi_travail.loc[df_annees].copy()

# Vérifier les années restantes
print(sorted(df_respi_travail["annee"].unique()))

[np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


2. Aggregats territoriaux
a) Verifications

Comme le jeu de données contient trois niveaux d’agrégation territoriale : départemental, régional et national. Les codes dept = 999 et region ≠ 99 semblent correspondre aux totaux régionaux, tandis que dept = 999 et region = 99 représenteraient le total national. Il faut distinguer ces niveaux lors des analyses pour éviter les doubles comptages.

In [245]:
#1. Controler 
# Examiner les couples département–région: afficher chaque association departement-region avec le nombre de ligne correspondant

# Copie pour les contrôles ; codes conservés comme chaînes
df_controle = df_respi_travail.copy()

for col in ["dept", "region"]:
    df_controle[col] = df_controle[col].astype("string").str.strip()  #transformer en texte et retierer les espaces eventuelles

couples = (
    df_controle.groupby(["dept", "region"], dropna=False)       #rassemble les lignes ayant le même couple département–région, on garde les lignes avec les NA
    .size()                                                     #compte les lignes de chaque groupes
    .reset_index(name="nombre_lignes")                          #resultats en DF avec 3 colonnes dept, region et nombre_lignes
)

display(couples.sort_values(["region", "dept"]))

# Examiner séparément les lignes supposées régionales et nationales
display(couples.loc[couples["dept"]=="999"])

,dept,region,nombre_lignes
96,971,1,252
101,999,1,252
75,75,11,252
77,77,11,252
78,78,11,252
...,...,...,...
117,999,93,252
28,2A,94,252
29,2B,94,252
118,999,94,252


,dept,region,nombre_lignes
101,999,1,252
102,999,11,252
103,999,2,252
104,999,24,252
105,999,27,252
106,999,28,252
107,999,3,252
108,999,32,252
109,999,4,252
110,999,44,252


La conclusion : les lignes dept = 999 doivent être mises à part pour une analyse départementale.

Verifications:
- chaque département est-il toujours associé à une seule région ?

In [246]:
#  chaque département est-il toujours associé à une seule région ?


departements = df_controle.loc[
    df_controle["dept"].notna()         #code de departement est renseigné et
    & (df_controle["dept"] != "999")     #le code département est différent de "999"
].copy()

regions_par_departement = (
    departements.groupby("dept")
    .agg(
        nombre_regions=("region", lambda s: s.nunique(dropna=False)),           #combien de valeurs différentes apparaissent dans region
        codes_region=("region", lambda s: s.drop_duplicates().tolist())         #la liste de ces valeurs, sans répétition
    )
)

# Une table vide signifie qu'aucun département n'a plusieurs régions
display(
    regions_par_departement.loc[
        regions_par_departement["nombre_regions"] != 1
    ]
)

# Repérer aussi les départements sans région ou associés à "99"
display(
    departements.loc[
        departements["region"].isna()
        | (departements["region"] == "99"),     #departements avec un region manquant ou egale à 99
        ["dept", "region"]
    ].drop_duplicates()
)

,nombre_regions,codes_region
dept,,


,dept,region


chaque departement est associé à une seule region, et il n'y a pas de departement non renseigné avec le code 99

--> comparer les totaux publiés aux sommes des départements, séparément pour chaque année 

In [247]:
# Selectionner tous age et tous sexe
base = df_controle.loc[
    (df_controle["cla_age_5"].astype("string") == "tsage")       #selectionner tous age et
    & (df_controle["sexe"].astype("string") == "9")              #tout sexe
].copy()


#Vérifier qu’une même observation n’apparaît pas plusieurs fois
cles = ["annee", "region" , "dept"]


doublons = base.duplicated(cles, keep=False)
display(base.loc[doublons])

if doublons.any():
    raise ValueError("Des doublons doivent être examinés avant de calculer les sommes.")


,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie


Resultat: La combinaison année + région + département est unique. Le tableu vide signifie que aucun doublon n'a ete trouvé

2. Aggregats territoriaux
b) Separation les 3 niveaux territoriaux

In [248]:
#Separer les 3 niveux territorieaux: les trois tables sont séparées pour que les totaux publiés ne soient jamais inclus dans la somme des départements.

# Lignes départementales
details = base.loc[
    base["dept"].notna()
    & (base["dept"] != "999")
    & base["region"].notna()
    & (base["region"] != "99")
].copy()

# Totaux régionaux supposés
totaux_regionaux = base.loc[
    (base["dept"] == "999")
    & base["region"].notna()
    & (base["region"] != "99")
].copy()

# Total national supposé
total_national = base.loc[
    (base["dept"] == "999")
    & (base["region"] == "99")
].copy()

In [249]:
#  Additionner les départements pour chaque année et région
sommes_regions = (
    details.groupby(["annee", "region"])[["Ntop", "Npop"]]
    .agg(lambda colonne: colonne.sum(skipna=False))
    .reset_index()
)

display(sommes_regions)

,annee,region,Ntop,Npop
0,2021,1,18480.0,393330
1,2021,11,574110.0,12599140
2,2021,2,15360.0,368880
3,2021,24,135010.0,2590760
4,2021,27,148220.0,2788190
...,...,...,...,...
67,2024,75,431870.0,6134910
68,2024,76,406520.0,6131500
69,2024,84,448260.0,8094330
70,2024,93,342540.0,5401550


In [250]:
#  Mettre les sommes et les totaux publiés côte à côte
controle_regions = sommes_regions.merge(
    totaux_regionaux[["annee", "region", "Ntop", "Npop"]],
    on=["annee", "region"],
    how="outer",
    suffixes=("_somme_departements", "_publie"),    #distingues les sommes calculés des valeurs publiés
    validate="one_to_one"                           #vérifie qu’il existe au maximum une ligne de chaque côté pour chaque année–région
)

# Calculer les differences

controle_regions["ecart_Ntop"] = (
    controle_regions["Ntop_somme_departements"]
    - controle_regions["Ntop_publie"]
)

controle_regions["ecart_Npop"] = (
    controle_regions["Npop_somme_departements"]
    - controle_regions["Npop_publie"]
)

display(controle_regions)
controle_regions.info()


,annee,region,Ntop_somme_departements,Npop_somme_departements,Ntop_publie,Npop_publie,ecart_Ntop,ecart_Npop
0,2021,1,18480.0,393330,18480.0,393330,0.0,0
1,2021,11,574110.0,12599140,574110.0,12599120,0.0,20
2,2021,2,15360.0,368880,15360.0,368880,0.0,0
3,2021,24,135010.0,2590760,135010.0,2590760,0.0,0
4,2021,27,148220.0,2788190,148210.0,2788190,10.0,0
...,...,...,...,...,...,...,...,...
67,2024,75,431870.0,6134910,431870.0,6134900,0.0,10
68,2024,76,406520.0,6131500,406500.0,6131500,20.0,0
69,2024,84,448260.0,8094330,448260.0,8094340,0.0,-10
70,2024,93,342540.0,5401550,342540.0,5401550,0.0,0


<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   annee                    72 non-null     int64  
 1   region                   72 non-null     string 
 2   Ntop_somme_departements  72 non-null     float64
 3   Npop_somme_departements  72 non-null     int64  
 4   Ntop_publie              72 non-null     float64
 5   Npop_publie              72 non-null     int64  
 6   ecart_Ntop               72 non-null     float64
 7   ecart_Npop               72 non-null     int64  
dtypes: float64(3), int64(4), string(1)
memory usage: 4.6 KB


In [251]:
#verification si il y a des lignes non calculables

colonnes_valeurs = [
    "Ntop_somme_departements", "Ntop_publie",
    "Npop_somme_departements", "Npop_publie"
]

lignes_na = controle_regions.loc[
    controle_regions[colonnes_valeurs].isna().any(axis=1)
]

display(lignes_na)

,annee,region,Ntop_somme_departements,Npop_somme_departements,Ntop_publie,Npop_publie,ecart_Ntop,ecart_Npop


pas de lignes avec des NA, tout est calculable

In [252]:
#afficher les valeurs non nulles 

lignes_ecarts = controle_regions.loc[
    (controle_regions["ecart_Ntop"].notna()
        & (controle_regions["ecart_Ntop"] != 0))
    |(controle_regions["ecart_Npop"].notna()
        & (controle_regions["ecart_Npop"] != 0)
    )
]

display(lignes_ecarts.sort_values(["annee", "region"])) 

lignes_ecarts.info()

,annee,region,Ntop_somme_departements,Npop_somme_departements,Ntop_publie,Npop_publie,ecart_Ntop,ecart_Npop
1,2021,11,574110.0,12599140,574110.0,12599120,0.0,20
4,2021,27,148220.0,2788190,148210.0,2788190,10.0,0
5,2021,28,207890.0,3349330,207900.0,3349330,-10.0,0
7,2021,32,369460.0,5996760,369460.0,5996750,0.0,10
9,2021,44,297500.0,5335930,297490.0,5335920,10.0,10
10,2021,52,189970.0,3919390,189960.0,3919380,10.0,10
11,2021,53,189740.0,3446250,189740.0,3446240,0.0,10
13,2021,75,365950.0,6168180,365940.0,6168160,10.0,20
14,2021,76,344440.0,6141180,344440.0,6141170,0.0,10
15,2021,84,377890.0,8191240,377900.0,8191240,-10.0,0


<class 'pandas.DataFrame'>
Index: 44 entries, 1 to 71
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   annee                    44 non-null     int64  
 1   region                   44 non-null     string 
 2   Ntop_somme_departements  44 non-null     float64
 3   Npop_somme_departements  44 non-null     int64  
 4   Ntop_publie              44 non-null     float64
 5   Npop_publie              44 non-null     int64  
 6   ecart_Ntop               44 non-null     float64
 7   ecart_Npop               44 non-null     int64  
dtypes: float64(3), int64(4), string(1)
memory usage: 3.1 KB


Resultat de verification de calcul: Sur 72 comparaisons région–année, 44 présentent un écart sur Ntop et/ou Npop entre la somme départementale et le total régional publié. L’origine de ces écarts n’a pas été déterminée. La suite de l’analyse utilise uniquement les observations départementales.

3. Pour la suite - creation de df_respi_departementale en gardeant la ##colonne region pour les regroupements eventuelles

In [253]:
df = df_respi_travail.copy()

# Harmoniser les codes territoriaux sans remplacer les NA
for col in ["dept", "region"]:
    df[col] = df[col].astype("string").str.strip()


# Attribuer un niveau territorial à chaque ligne
df["niveau"] = "a_verifier"

region_connue = df["region"].notna() & (df["region"] != "99")

#on donne la valeur "departement" aux lignes ou le departement est connu, pas egale à 999 et region est connu
df.loc[
    df["dept"].notna() & (df["dept"] !="999") & region_connue,
    "niveau"
] = "departements"

#on donne la valeur "regions" aux lignes ou le departement est  à 999 et region est connu
df.loc[
    df["dept"].eq("999") & region_connue,
    "niveau"
] = "regions"

#etiquette national
df.loc[
    (df["dept"] == "999") & (df["region"] == "99"),
    "niveau"
] = "national"



In [254]:
# Séparer les niveaux et exporter les CSV

# Séparer les niveaux et exporter les CSV dans un dossier
from pathlib import Path
dossier = Path("exports_effectifs")
dossier.mkdir(exist_ok=True)

# Séparer les données en 4 DataFrames
df_respi_dept = df[df["niveau"] == "departements"].copy()
df_respi_region = df[df["niveau"] == "regions"].copy()
df_respi_national = df[df["niveau"] == "national"].copy()
df_respi_a_verifier = df[df["niveau"] == "a_verifier"].copy()

display(df_respi_dept.head(5))
display(df_respi_region.head(5))
display(df_respi_national.head(5))
display(df_respi_a_verifier.head(5))


#exporter les csv

df_respi_dept.to_csv("respi_dept.csv", index=False, sep=",")
df_respi_region.to_csv("respi_region.csv", index=False, sep=",")
df_respi_national.to_csv("respi_national.csv", index=False, sep=",")
df_respi_a_verifier.to_csv("respi_a_verifier.csv", index=False, sep=",")



,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie,niveau
38778,2024,RES_CAT_EXC,85-89,1,32,02,700.0,3770,18.503,Maladie_Respi_Chronique,departements
38780,2024,RES_CAT_EXC,85-89,1,44,10,350.0,2620,13.354,Maladie_Respi_Chronique,departements
38781,2024,RES_CAT_EXC,85-89,1,44,52,280.0,1770,15.971,Maladie_Respi_Chronique,departements
38782,2024,RES_CAT_EXC,85-89,1,44,67,1270.0,8750,14.485,Maladie_Respi_Chronique,departements
38783,2024,RES_CAT_EXC,85-89,1,44,08,390.0,2190,17.986,Maladie_Respi_Chronique,departements


,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie,niveau
38777,2024,RES_CAT_EXC,85-89,1,28,999,4470.0,27770,16.100,Maladie_Respi_Chronique,regions
38779,2024,RES_CAT_EXC,85-89,1,32,999,6710.0,37310,17.975,Maladie_Respi_Chronique,regions
38791,2024,RES_CAT_EXC,85-89,1,75,999,8720.0,60560,14.398,Maladie_Respi_Chronique,regions
38795,2024,RES_CAT_EXC,85-89,1,76,999,8270.0,56370,14.670,Maladie_Respi_Chronique,regions
38801,2024,RES_CAT_EXC,85-89,1,93,999,7050.0,48920,14.405,Maladie_Respi_Chronique,regions


,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie,niveau
38803,2024,RES_CAT_EXC,85-89,1,99,999,78060.0,547950,14.246,Maladie_Respi_Chronique,national
38841,2024,RES_CAT_EXC,85-89,2,99,999,96740.0,865950,11.171,Maladie_Respi_Chronique,national
39037,2024,RES_CAT_EXC,95et+,1,99,999,9680.0,71430,13.550,Maladie_Respi_Chronique,national
39077,2024,RES_CAT_EXC,95et+,2,99,999,24950.0,243160,10.262,Maladie_Respi_Chronique,national
39155,2024,RES_CAT_EXC,tsage,1,99,999,2031930.0,32440780,6.263,Maladie_Respi_Chronique,national


,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie,niveau


Verification le df_respi_dept

In [255]:
df_respi_dept.info()

<class 'pandas.DataFrame'>
Index: 25452 entries, 38778 to 5709338
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   annee       25452 non-null  int64  
 1   top         25452 non-null  str    
 2   cla_age_5   25452 non-null  str    
 3   sexe        25452 non-null  int64  
 4   region      25452 non-null  string 
 5   dept        25452 non-null  string 
 6   Ntop        25418 non-null  float64
 7   Npop        25452 non-null  int64  
 8   prev        25418 non-null  float64
 9   pathologie  25452 non-null  str    
 10  niveau      25452 non-null  str    
dtypes: float64(2), int64(3), str(4), string(2)
memory usage: 2.3 MB


In [256]:
df_respi_dept.isna().sum()

annee          0
top            0
cla_age_5      0
sexe           0
region         0
dept           0
Ntop          34
Npop           0
prev          34
pathologie     0
niveau         0
dtype: int64

In [257]:
df_respi_dept.nunique()

annee             4
top               1
cla_age_5        21
sexe              3
region           18
dept            101
Ntop           1727
Npop           7731
prev          11233
pathologie        1
niveau            1
dtype: int64

In [258]:
print(df_respi_dept["dept"].unique().tolist())

['02', '10', '52', '67', '08', '44', '49', '53', '85', '40', '47', '86', '12', '30', '65', '15', '42', '63', '69', '07', '2A', '971', '972', '973', '77', '91', '92', '95', '28', '21', '58', '71', '27', '88', '72', '56', '16', '33', '64', '79', '31', '32', '01', '26', '73', '05', '75', '93', '36', '45', '89', '90', '76', '80', '55', '35', '17', '81', '82', '09', '38', '43', '83', '974', '976', '78', '18', '37', '70', '51', '54', '57', '19', '24', '34', '74', '13', '2B', '39', '59', '87', '48', '04', '84', '50', '68', '29', '46', '66', '94', '14', '03', '06', '41', '60', '62', '22', '11', '25', '61', '23']


df_respi_dept contient encore des agregats tous age et tout sexe!

Est-ce qu'on veut faire un prediction globale par departement ou par departement et par classe d'age? 

4. a) Creation de df_respi_dept_global_metropole pour un prediction global (on garde tous age et tout sexe dans le df, sans distinguer les categores d'ages et homme/femme

In [259]:
#Creation de df pour une prediciton globale: on garde tous age et tous sexe dans le df

df_respi_dept_global_metropole = df_respi_dept[
    (df_respi_dept["cla_age_5"] == "tsage") &
    (df_respi_dept["sexe"] == 9) &
    (~df_respi_dept["dept"].isin(["971", "972", "973", "974", "976"]))
].copy()

In [260]:
#verifier les valeurs manquantes
df_respi_dept_global_metropole[["Npop", "prev"]].isna().sum()

Npop    0
prev    0
dtype: int64

In [261]:
#verifier si j'ai bien une seule ligne par departement et par semaine
df_respi_dept_global_metropole.duplicated(subset=["dept", "annee"]).sum()

np.int64(0)

In [262]:
print(df_respi_dept_global_metropole.shape)
print(df_respi_dept_global_metropole[["Npop", "prev"]].isna().sum())

(384, 11)
Npop    0
prev    0
dtype: int64


le df_respi_dept_global_metropole est pret pour la jointure (sauf voir comment faire pour les années)

In [263]:
#extraction fichier 
df_respi_dept_global_metropole.to_csv("respi_dept_global_metropole.csv", index=False, sep=",")

4. b) Creation de df pour une prediction par classe d'age - si pertinant
 -- la question: est-ce qu'on peut recalculer un prevalence pour les categories d'ages regrouper sans creer un biais? 

In [264]:
#df avec tous sexe et avec tous les categories d'age sauf tous age
df_respi_dept_metropole = df_respi_dept[
    (df_respi_dept["cla_age_5"] != "tsage") &
    (df_respi_dept["sexe"] == 9) &
    (~df_respi_dept["dept"].isin(["971", "972", "973", "974", "976"]))
].copy()

print("Lignes du fichier df_respi_dept :", len(df_respi_dept))
print("Lignes métropolitaines :", len(df_respi_dept_metropole))

Lignes du fichier df_respi_dept : 25452
Lignes métropolitaines : 7680


Objectif est de vérifier si tu peux regrouper les 20 classes d'âge en 3 catégories (0-14, 15-64, 65+) et recalculer prev à partir de Ntop et Npop.

In [265]:
# Créer une nouvelle colonne pour les 3 catégories

df_age = df_respi_dept_metropole.copy()


df_age["age_groupe"] = "65+"

df_age.loc[df_age["cla_age_5"].isin(["00-04", "05-09", "10-14"]), "age_groupe"] = "0-14"

df_age.loc[df_age["cla_age_5"].isin([
    "15-19", "20-24", "25-29", "30-34", "35-39",
    "40-44", "45-49", "50-54", "55-59", "60-64"
]), "age_groupe"] = "15-64"


In [285]:
display(df_age[["Ntop", "Npop"]].isna().sum())
display(df_age.head(5))
display(df_age.info())


Ntop    0
Npop    0
dtype: int64

,annee,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,pathologie,niveau,age_groupe
38845,2024,RES_CAT_EXC,85-89,9,11,75,4240.0,36280,11.689,Maladie_Respi_Chronique,departements,65+
38846,2024,RES_CAT_EXC,85-89,9,11,91,2500.0,20870,11.987,Maladie_Respi_Chronique,departements,65+
38847,2024,RES_CAT_EXC,85-89,9,11,93,2430.0,17170,14.150,Maladie_Respi_Chronique,departements,65+
38848,2024,RES_CAT_EXC,85-89,9,11,95,2190.0,16060,13.641,Maladie_Respi_Chronique,departements,65+
38849,2024,RES_CAT_EXC,85-89,9,24,36,700.0,6550,10.607,Maladie_Respi_Chronique,departements,65+


<class 'pandas.DataFrame'>
Index: 7680 entries, 38845 to 5709326
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   annee       7680 non-null   int64  
 1   top         7680 non-null   str    
 2   cla_age_5   7680 non-null   str    
 3   sexe        7680 non-null   int64  
 4   region      7680 non-null   string 
 5   dept        7680 non-null   string 
 6   Ntop        7680 non-null   float64
 7   Npop        7680 non-null   int64  
 8   prev        7680 non-null   float64
 9   pathologie  7680 non-null   str    
 10  niveau      7680 non-null   str    
 11  age_groupe  7680 non-null   str    
dtypes: float64(2), int64(3), str(5), string(2)
memory usage: 780.0 KB


None

In [303]:
#regrouper et recalculer la prevalence dans le df avec les categories regroupées par an et par dept
df_respi_age = (
    df_age.groupby(["annee", "dept", "age_groupe", "sexe", "pathologie"])[["Ntop", "Npop"]]
    .sum(min_count=1)
    .reset_index()
)

df_respi_age["prev_calc"] = df_respi_age["Ntop"] / df_respi_age["Npop"] * 100

display(df_respi_age.head())

display(ecarts.abs().max())
display(ecarts.abs().idxmax())

display(df_respi_dept[["Npop", "Ntop"]].max())

,annee,dept,age_groupe,sexe,pathologie,Ntop,Npop,prev_calc
0,2021,01,0-14,9,Maladie_Respi_Chronique,4330.0,119000,3.638655
1,2021,01,15-64,9,Maladie_Respi_Chronique,12970.0,399590,3.245827
2,2021,01,65+,9,Maladie_Respi_Chronique,11740.0,122010,9.622162
3,2021,02,0-14,9,Maladie_Respi_Chronique,3570.0,91960,3.882123
4,2021,02,15-64,9,Maladie_Respi_Chronique,16790.0,319770,5.250649


Ntop    50.0
Npop    50.0
dtype: float64

Ntop    (2021, 91)
Npop    (2022, 79)
dtype: object

Npop    2594820.0
Ntop     178170.0
dtype: float64

In [304]:
# Somme des 3 groupes par département et année
somme = df_respi_age.groupby(["annee", "dept"])[["Ntop", "Npop"]].sum()

# Effectifs officiels tous âges
total = df_respi_dept[
    (df_respi_dept["cla_age_5"] == "tsage") &
    (df_respi_dept["sexe"].astype(str) == "9") &
    (~df_respi_dept["dept"].isin(["971", "972", "973", "974", "976"]))
].set_index(["annee", "dept"])[["Ntop", "Npop"]]

# Comparaison
ecarts = somme - total
display(ecarts)

# Garder les lignes avec un écart non nul (hors valeurs manquantes)
masque = (ecarts.notna() & ecarts.ne(0)).any(axis=1)
ecarts_non_nuls = ecarts.loc[masque].copy()

# Trier du plus grand au plus petit écart en valeur absolue
ordre = ecarts_non_nuls.abs().max(axis=1).sort_values(ascending=False).index
ecarts_non_nuls = ecarts_non_nuls.loc[ordre]

print(f"{len(ecarts_non_nuls)} lignes avec au moins un écart différent de 0")
display(ecarts_non_nuls)




Ntop  Npop
annee dept            
2021  01    10.0    30
      02     0.0   -10
      03    10.0    20
      04     0.0     0
      05     0.0    30
...          ...   ...
2024  91    10.0    20
      92     0.0     0
      93    10.0     0
      94    20.0     0
      95    20.0    30

[384 rows x 2 columns]

363 lignes avec au moins un écart différent de 0


Ntop  Npop
annee dept            
2021  91    50.0    20
2022  79    30.0    50
2024  63    20.0    40
2021  55    10.0    40
2024  37    40.0    30
...          ...   ...
      59    10.0     0
      66   -10.0    10
      93    10.0     0
2021  08     0.0    10
      02     0.0   -10

[363 rows x 2 columns]

In [305]:
df_respi_age.info()

<class 'pandas.DataFrame'>
RangeIndex: 1152 entries, 0 to 1151
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   annee       1152 non-null   int64  
 1   dept        1152 non-null   string 
 2   age_groupe  1152 non-null   str    
 3   sexe        1152 non-null   int64  
 4   pathologie  1152 non-null   str    
 5   Ntop        1152 non-null   float64
 6   Npop        1152 non-null   int64  
 7   prev_calc   1152 non-null   float64
dtypes: float64(2), int64(3), str(2), string(1)
memory usage: 72.1 KB


In [299]:
#Quel est la valeur Ntop et Npop pour les ecartes maximales?

for colonne in ["Ntop", "Npop"]:
    position = ecarts[colonne].abs().idxmax()

    print(f"\nÉcart maximal de {colonne} : {position}")

    comparaison = pd.DataFrame({
        "Somme des groupes": somme.loc[position],
        "Total officiel": total.loc[position],
        "Écart": ecarts.loc[position]
    })

    comparaison["Écart (%)"] = (
        comparaison["Écart"]
        / comparaison["Total officiel"].replace(0, float("nan"))
        * 100
    ).round(2)

    display(comparaison)


Écart maximal de Ntop : (np.int64(2021), '91')


,Somme des groupes,Total officiel,Écart,Écart (%)
Ntop,60950.0,60900.0,50.0,0.08
Npop,1352360.0,1352340.0,20.0,0.00



Écart maximal de Npop : (np.int64(2022), '79')


,Somme des groupes,Total officiel,Écart,Écart (%)
Ntop,22950.0,22920.0,30.0,0.13
Npop,379330.0,379280.0,50.0,0.01


In [300]:
#Plus grand ecart en %

# Pourcentage par rapport au total officiel
ecarts_pct = ecarts.div(total.replace(0, float("nan"))) * 100

# Tableau complet : écarts et pourcentages
comparaison = ecarts.join(ecarts_pct.add_suffix("_pct"))

display(comparaison)
for colonne in ["Ntop", "Npop"]:
    position = ecarts_pct[colonne].abs().idxmax()

    print(f"\nPlus grand écart en % pour {colonne} : {position}")
    display(comparaison.loc[[position]])

Ntop  Npop  Ntop_pct  Npop_pct
annee dept                                
2021  01    10.0    30  0.034447  0.004683
      02     0.0   -10  0.000000 -0.001902
      03    10.0    20  0.055772  0.005991
      04     0.0     0  0.000000  0.000000
      05     0.0    30  0.000000  0.020664
...          ...   ...       ...       ...
2024  91    10.0    20  0.013933  0.001503
      92     0.0     0  0.000000  0.000000
      93    10.0     0  0.010967  0.000000
      94    20.0     0  0.026961  0.000000
      95    20.0    30  0.026846  0.002311

[384 rows x 4 columns]


Plus grand écart en % pour Ntop : (np.int64(2022), '48')


,,Ntop,Npop,Ntop_pct,Npop_pct
annee,dept,,,,
2022,48,30.0,20,0.731707,0.027027



Plus grand écart en % pour Npop : (np.int64(2024), '48')


,,Ntop,Npop,Ntop_pct,Npop_pct
annee,dept,,,,
2024,48,0.0,-20,0.0,-0.027503


363 lignes sur 1152 contiennten au moins un valeurs differentes de 0 (ecartes entre prev publié et calculé). Le maximum ecart est 50 pour Npop et Ntop, et la max valeur de Npop est 2594820.0 , Ntop est 178170.0 . Le maximum ecart observé pour Ntop est de 0.73% et pour Npop est -0.02% qui restent assez petit pour que ca soit acceptablent

#Verification supplementaire pour le df_respi_dept pour identifier la provenence des lignes contenant des valeurs manquantes

In [272]:
#verification si les lignes NAs appartiennent au lignes hommes/femmes

df_respi_dept[["prev", "Ntop", "Npop"]].isna().sum()

prev    34
Ntop    34
Npop     0
dtype: int64

In [273]:
df_test1 = df_respi_dept[
    (df_respi_dept["cla_age_5"] != "tsage") 
].copy()

df_test1[["prev", "Ntop", "Npop"]].isna().sum()

prev    34
Ntop    34
Npop     0
dtype: int64

In [274]:
# Répartition des lignes avec/sans NA selon le sexe
for colonne in ["prev", "Ntop"]:
    print(f"\nValeurs manquantes dans {colonne}")

    display(pd.crosstab(
        df_test1["sexe"].fillna("Sexe manquant"),
        df_test1[colonne].isna().map({
            False: "Valeur présente",
            True: "Valeur manquante"
        })
    ))


Valeurs manquantes dans prev


prev,Valeur manquante,Valeur présente
sexe,,
1,18,8062
2,12,8068
9,4,8076



Valeurs manquantes dans Ntop


Ntop,Valeur manquante,Valeur présente
sexe,,
1,18,8062
2,12,8068
9,4,8076


Verification quel filtre enleve les valeurs manquantes pour arriver à 0 dans df_respi_dept_metropole et df_respi_dept_global_metropole

In [275]:
df_test2 = df_respi_dept[
     (~df_respi_dept["dept"].isin(["971", "972", "973", "974", "976"]))
].copy()

df_test2[["prev", "Ntop", "Npop"]].isna().sum()

prev    2
Ntop    2
Npop    0
dtype: int64

In [276]:
# Répartition des lignes avec/sans NA selon departement
for colonne in ["prev", "Ntop"]:
    print(f"\nValeurs manquantes dans {colonne}")

    tableau = pd.crosstab(
        df_test2["dept"].fillna("dept manquant"),
        df_test2[colonne].isna().map({
            False: "Valeur présente",
            True: "Valeur manquante"
        })
    )

    if "Valeur manquante" in tableau.columns:
        display(tableau.loc[tableau["Valeur manquante"] > 0])
    else:
        print("Aucune valeur manquante.")


Valeurs manquantes dans prev


prev,Valeur manquante,Valeur présente
dept,,
48,1,251
90,1,251



Valeurs manquantes dans Ntop


Ntop,Valeur manquante,Valeur présente
dept,,
48,1,251
90,1,251


In [277]:
df_test3 = df_respi_dept[
     (df_respi_dept["cla_age_5"] !="9")
].copy()

df_test3[["prev", "Ntop", "Npop"]].isna().sum()


prev    34
Ntop    34
Npop     0
dtype: int64

In [278]:
for colonne in ["prev", "Ntop"]:
    print(f"\nValeurs manquantes dans {colonne}")

    tableau = pd.crosstab(
        df_test3["cla_age_5"].fillna("age manquant"),
        df_test3[colonne].isna().map({
            False: "Valeur présente",
            True: "Valeur manquante"
        })
    )

    if "Valeur manquante" in tableau.columns:
        display(tableau.loc[tableau["Valeur manquante"] > 0])
    else:
        print("Aucune valeur manquante.")


Valeurs manquantes dans prev


prev,Valeur manquante,Valeur présente
cla_age_5,,
85-89,1,1211
90-94,12,1200
95et+,21,1191



Valeurs manquantes dans Ntop


Ntop,Valeur manquante,Valeur présente
cla_age_5,,
85-89,1,1211
90-94,12,1200
95et+,21,1191


5. Pivot de fichier global (df_respi_dept_global_metropole.head) pour rapprochement avec des données cibles --> fichier exploitable
df_respi_global_metropol : données par departement, tous age, tous sexe, sans DROM

In [295]:
#renommer la colonne dept en code_dept
df_respi_dept_global_metropole.rename(columns={'dept': 'code_dept'}, inplace=True)
display(df_respi_dept_global_metropole.head(5))
display(df_respi_dept_global_metropole.info())
display(df_respi_dept_global_metropole.nunique())

,annee,top,cla_age_5,sexe,region,code_dept,Ntop,Npop,prev,pathologie,niveau
39205,2024,RES_CAT_EXC,tsage,9,11,75,101510.0,2068810,4.907,Maladie_Respi_Chronique,departements
39206,2024,RES_CAT_EXC,tsage,9,11,77,87380.0,1450940,6.022,Maladie_Respi_Chronique,departements
39207,2024,RES_CAT_EXC,tsage,9,11,91,71770.0,1331090,5.392,Maladie_Respi_Chronique,departements
39208,2024,RES_CAT_EXC,tsage,9,11,95,74500.0,1298030,5.739,Maladie_Respi_Chronique,departements
39209,2024,RES_CAT_EXC,tsage,9,24,18,16990.0,289740,5.862,Maladie_Respi_Chronique,departements


<class 'pandas.DataFrame'>
Index: 384 entries, 39205 to 5556836
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   annee       384 non-null    int64  
 1   top         384 non-null    str    
 2   cla_age_5   384 non-null    str    
 3   sexe        384 non-null    int64  
 4   region      384 non-null    string 
 5   code_dept   384 non-null    string 
 6   Ntop        384 non-null    float64
 7   Npop        384 non-null    int64  
 8   prev        384 non-null    float64
 9   pathologie  384 non-null    str    
 10  niveau      384 non-null    str    
dtypes: float64(2), int64(3), str(4), string(2)
memory usage: 36.0 KB


None

annee           4
top             1
cla_age_5       1
sexe            1
region         13
code_dept      96
Ntop          371
Npop          383
prev          358
pathologie      1
niveau          1
dtype: int64

In [280]:
#1er tableau avec tsage et tout sex

df_pivot_1 = df_respi_dept_global_metropole.pivot(
    index=["annee", "code_dept"],              #définit une ligne du résultat : une combinaison semaine/département.
    columns=["pathologie", "cla_age_5", "sexe"],             #transforme les combinaisons pathologie/âge en colonnes.
    values=["Ntop", "Npop", "prev"]    #indique quelles valeurs placer dans ces colonnes : les deux taux
)

display(df_pivot_1.head())
print("Dimensions :", df_pivot_1.shape)

Ntop                    Npop  \
pathologie      Maladie_Respi_Chronique Maladie_Respi_Chronique   
cla_age_5                         tsage                   tsage   
sexe                                  9                       9   
annee code_dept                                                   
2021  01                        29030.0                640570.0   
      02                        35010.0                525790.0   
      03                        17930.0                333820.0   
      04                         9450.0                169250.0   
      05                         6900.0                145180.0   

                                   prev  
pathologie      Maladie_Respi_Chronique  
cla_age_5                         tsage  
sexe                                  9  
annee code_dept                          
2021  01                          4.532  
      02                          6.658  
      03                          5.372  
      04                          5.583  
      05                          4.750

Dimensions : (384, 3)


In [281]:
# Fusionner les niveaux de colonnes en un seul nom
df_pivot_1.columns = [
    "_".join(map(str, colonne))
    for colonne in df_pivot_1.columns
]

# Mettre année et département dans les colonnes
df_pivot_1 = df_pivot_1.reset_index()

display(df_pivot_1.head())
print("Dimensions :", df_pivot_1.shape)

,annee,code_dept,Ntop_Maladie_Respi_Chronique_tsage_9,Npop_Maladie_Respi_Chronique_tsage_9,prev_Maladie_Respi_Chronique_tsage_9
0,2021,01,29030.0,640570.0,4.532
1,2021,02,35010.0,525790.0,6.658
2,2021,03,17930.0,333820.0,5.372
3,2021,04,9450.0,169250.0,5.583
4,2021,05,6900.0,145180.0,4.750


Dimensions : (384, 5)


In [282]:
#Exporter le fichier

df_pivot_1.to_csv(
    "effectif_metropole_2021_2024_tsage_toussexe_respiratoire.csv",
    index=False,
    )

6. 2 ieme tableau pivot information par année par departement et categorie d'age par colonnes pour pathologie maladies respiratoires chronqiues (toujours tous sexe)

In [306]:
df_respi_age.rename(columns={'dept': 'code_dept'}, inplace=True)
display(df_respi_age.head(5))
display(df_respi_age.info())
display(df_respi_age.nunique())

,annee,code_dept,age_groupe,sexe,pathologie,Ntop,Npop,prev_calc
0,2021,01,0-14,9,Maladie_Respi_Chronique,4330.0,119000,3.638655
1,2021,01,15-64,9,Maladie_Respi_Chronique,12970.0,399590,3.245827
2,2021,01,65+,9,Maladie_Respi_Chronique,11740.0,122010,9.622162
3,2021,02,0-14,9,Maladie_Respi_Chronique,3570.0,91960,3.882123
4,2021,02,15-64,9,Maladie_Respi_Chronique,16790.0,319770,5.250649


<class 'pandas.DataFrame'>
RangeIndex: 1152 entries, 0 to 1151
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   annee       1152 non-null   int64  
 1   code_dept   1152 non-null   string 
 2   age_groupe  1152 non-null   str    
 3   sexe        1152 non-null   int64  
 4   pathologie  1152 non-null   str    
 5   Ntop        1152 non-null   float64
 6   Npop        1152 non-null   int64  
 7   prev_calc   1152 non-null   float64
dtypes: float64(2), int64(3), str(2), string(1)
memory usage: 72.1 KB


None

annee            4
code_dept       96
age_groupe       3
sexe             1
pathologie       1
Ntop           923
Npop          1129
prev_calc     1152
dtype: int64

In [314]:
#2ieme tableau avec tsage et tout sex

df_pivot_2 = df_respi_age.pivot(
    index=["annee", "code_dept"],              #définit une ligne du résultat : une combinaison semaine/département.
    columns=["pathologie", "age_groupe", "sexe"],             #transforme les combinaisons pathologie/âge en colonnes.
    values=["Ntop", "Npop", "prev_calc"]    #indique quelles valeurs placer dans ces colonnes : les deux taux
)

display(df_pivot_2.head())
print("Dimensions :", df_pivot_2.shape)

Ntop                    \
pathologie      Maladie_Respi_Chronique                     
age_groupe                         0-14    15-64      65+   
sexe                                  9        9        9   
annee code_dept                                             
2021  01                         4330.0  12970.0  11740.0   
      02                         3570.0  16790.0  14650.0   
      03                         2080.0   7530.0   8330.0   
      04                          780.0   3950.0   4720.0   
      05                          640.0   2810.0   3450.0   

                                   Npop                      \
pathologie      Maladie_Respi_Chronique                       
age_groupe                         0-14     15-64       65+   
sexe                                  9         9         9   
annee code_dept                                               
2021  01                       119000.0  399590.0  122010.0   
      02                        91960.0  319770.0  114050.0   
      03                        47400.0  190880.0   95560.0   
      04                        24390.0   99870.0   44990.0   
      05                        21020.0   87130.0   37060.0   

                              prev_calc                       
pathologie      Maladie_Respi_Chronique                       
age_groupe                         0-14     15-64        65+  
sexe                                  9         9          9  
annee code_dept                                               
2021  01                       3.638655  3.245827   9.622162  
      02                       3.882123  5.250649  12.845243  
      03                       4.388186  3.944887   8.717036  
      04                       3.198032  3.955142  10.491220  
      05                       3.044719  3.225066   9.309228

Dimensions : (384, 9)


In [315]:
# Fusionner les niveaux de colonnes en un seul nom
df_pivot_2.columns = [
    "_".join(map(str, colonne))
    for colonne in df_pivot_2.columns
]

# Mettre année et département dans les colonnes
df_pivot_2 = df_pivot_2.reset_index()

display(df_pivot_2.head())
print("Dimensions :", df_pivot_2.shape)

,annee,code_dept,Ntop_Maladie_Respi_Chronique_0-14_9,Ntop_Maladie_Respi_Chronique_15-64_9,Ntop_Maladie_Respi_Chronique_65+_9,Npop_Maladie_Respi_Chronique_0-14_9,Npop_Maladie_Respi_Chronique_15-64_9,Npop_Maladie_Respi_Chronique_65+_9,prev_calc_Maladie_Respi_Chronique_0-14_9,prev_calc_Maladie_Respi_Chronique_15-64_9,prev_calc_Maladie_Respi_Chronique_65+_9
0,2021,01,4330.0,12970.0,11740.0,119000.0,399590.0,122010.0,3.638655,3.245827,9.622162
1,2021,02,3570.0,16790.0,14650.0,91960.0,319770.0,114050.0,3.882123,5.250649,12.845243
2,2021,03,2080.0,7530.0,8330.0,47400.0,190880.0,95560.0,4.388186,3.944887,8.717036
3,2021,04,780.0,3950.0,4720.0,24390.0,99870.0,44990.0,3.198032,3.955142,10.491220
4,2021,05,640.0,2810.0,3450.0,21020.0,87130.0,37060.0,3.044719,3.225066,9.309228


Dimensions : (384, 11)


In [316]:
#Exporter le fichier

df_pivot_2.to_csv(
    "effectif_metropole_2021_2024_3catAge_toussexe_respiratoire.csv",
    index=False,
    )

7. Fusionner les 2 fichiers

In [317]:
df_fusion = df_pivot_1.merge(
    df_pivot_2,
    on=["annee", "code_dept"],
    how="outer",
    validate="one_to_one",
    suffixes=("_1", "_2")
)

print(df_fusion.shape)
display(df_fusion.head())
display(df_fusion.info())
display(df_fusion.nunique())

(384, 14)


,annee,code_dept,Ntop_Maladie_Respi_Chronique_tsage_9,Npop_Maladie_Respi_Chronique_tsage_9,prev_Maladie_Respi_Chronique_tsage_9,Ntop_Maladie_Respi_Chronique_0-14_9,Ntop_Maladie_Respi_Chronique_15-64_9,Ntop_Maladie_Respi_Chronique_65+_9,Npop_Maladie_Respi_Chronique_0-14_9,Npop_Maladie_Respi_Chronique_15-64_9,Npop_Maladie_Respi_Chronique_65+_9,prev_calc_Maladie_Respi_Chronique_0-14_9,prev_calc_Maladie_Respi_Chronique_15-64_9,prev_calc_Maladie_Respi_Chronique_65+_9
0,2021,01,29030.0,640570.0,4.532,4330.0,12970.0,11740.0,119000.0,399590.0,122010.0,3.638655,3.245827,9.622162
1,2021,02,35010.0,525790.0,6.658,3570.0,16790.0,14650.0,91960.0,319770.0,114050.0,3.882123,5.250649,12.845243
2,2021,03,17930.0,333820.0,5.372,2080.0,7530.0,8330.0,47400.0,190880.0,95560.0,4.388186,3.944887,8.717036
3,2021,04,9450.0,169250.0,5.583,780.0,3950.0,4720.0,24390.0,99870.0,44990.0,3.198032,3.955142,10.491220
4,2021,05,6900.0,145180.0,4.750,640.0,2810.0,3450.0,21020.0,87130.0,37060.0,3.044719,3.225066,9.309228


<class 'pandas.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 14 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   annee                                      384 non-null    int64  
 1   code_dept                                  384 non-null    string 
 2   Ntop_Maladie_Respi_Chronique_tsage_9       384 non-null    float64
 3   Npop_Maladie_Respi_Chronique_tsage_9       384 non-null    float64
 4   prev_Maladie_Respi_Chronique_tsage_9       384 non-null    float64
 5   Ntop_Maladie_Respi_Chronique_0-14_9        384 non-null    float64
 6   Ntop_Maladie_Respi_Chronique_15-64_9       384 non-null    float64
 7   Ntop_Maladie_Respi_Chronique_65+_9         384 non-null    float64
 8   Npop_Maladie_Respi_Chronique_0-14_9        384 non-null    float64
 9   Npop_Maladie_Respi_Chronique_15-64_9       384 non-null    float64
 10  Npop_Maladie_Respi_Chronique_65+_9   

None

annee                                          4
code_dept                                     96
Ntop_Maladie_Respi_Chronique_tsage_9         371
Npop_Maladie_Respi_Chronique_tsage_9         383
prev_Maladie_Respi_Chronique_tsage_9         358
Ntop_Maladie_Respi_Chronique_0-14_9          321
Ntop_Maladie_Respi_Chronique_15-64_9         367
Ntop_Maladie_Respi_Chronique_65+_9           364
Npop_Maladie_Respi_Chronique_0-14_9          379
Npop_Maladie_Respi_Chronique_15-64_9         379
Npop_Maladie_Respi_Chronique_65+_9           381
prev_calc_Maladie_Respi_Chronique_0-14_9     384
prev_calc_Maladie_Respi_Chronique_15-64_9    384
prev_calc_Maladie_Respi_Chronique_65+_9      384
dtype: int64

In [318]:
#Exporter le fichier

df_fusion.to_csv(
    "effectif_metropole_2021_2024_respiratoire_exploitable.csv",
    index=False,
    )